# 薪資預測 API 教學：實作 `/train` 線上訓練與模型更新端點

> 對象：修習多元線性迴歸 / 模型部署課程的學生
> 對應程式碼：`app.py` 第 172~189 行的 `train_api` 函數
>
> 這份 Notebook 會帶你**一步步拆解**線上模型重新訓練與即時更新機制，並用 `TestClient` 模擬完整呼叫流程。

## 學習目標

- [ ] 理解「線上重訓 (Online Retraining)」的概念與應用場景
- [ ] 看懂 FastAPI 訓練端點的請求與回應模型（`TrainConfig` 與 `TrainResult`）
- [ ] 掌握 `train_and_save_model()` 如何依超參數重新訓練並覆蓋 `.joblib` 模型檔
- [ ] 理解 `load_model_state()` 重新載入全域狀態（`MODEL_STATE`）的必要性
- [ ] 能使用 `TestClient` 發送 POST `/train` 請求，並驗證模型更新後對 `/predict` 的影響

## 背景：`/train` 端點運作流程

在現實部署環境中，當我們想要嘗試不同的模型演算法（如 Ridge 嶺迴歸、Lasso 迴歸）、調整超參數（如正則化強度 `alpha`）或改變訓練集比例時，我們希望**不用關閉或重啟伺服器**就能完成模型更新。

`/train` 端點的內部三大核心步驟：

```
[收到 TrainConfig 請求]
       │
       ▼
1. train_and_save_model(...)  ──> 重新訓練模型，覆蓋 salary_model.joblib
       │
       ▼
2. load_model_state()         ──> 重新讀取 joblib 檔，更新全域變數 MODEL_STATE
       │
       ▼
3. 回傳 TrainResult(**res)     ──> 回傳 R² 得分、特徵權重、截距與訓練耗時
```

接下來我們一步步拆解並動手實作。

In [ ]:
# ============================================
# 0. 載入套件與環境設定
# ============================================

import os, sys, time
import joblib
import pandas as pd
import numpy as np
from pprint import pprint

current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# 匯入訓練邏輯模組
from train_save import train_and_save_model

print("環境準備完畢，已成功載入 train_and_save_model 模組。")

---
## Part 1：定義請求與回應的 Pydantic 模型

在 `app.py` 中，訓練端點需要接收使用者傳遞的訓練參數，並回傳格式化的訓練結果：

- **`TrainConfig`**（請求模型）：包含 `test_size` (0.1~0.5)、`random_state` (>=0)、`model_type` (LinearRegression, Lasso, Ridge) 以及 `alpha` (正則化強度)。
- **`TrainResult`**（回應模型）：回傳 `status`、`r2`、`coef`、`intercept`、`feature_coefs`、`model_type`、`alpha`、`train_time` 與 `message`。

In [ ]:
# ============================================
# 定義 Pydantic 訓練相關模型（與 app.py 相同）
# ============================================

from pydantic import BaseModel, Field

class TrainConfig(BaseModel):
    test_size: float = Field(0.2, description="測試集分割比例", ge=0.1, le=0.5)
    random_state: int = Field(76, description="隨機種子", ge=0)
    model_type: str = Field("LinearRegression", description="模型演算法類型 (LinearRegression, Lasso, Ridge)")
    alpha: float = Field(1.0, description="正則化強度 alpha (適用於 Lasso 與 Ridge)", ge=0.001, le=100.0)

class TrainResult(BaseModel):
    status: str = Field(..., description="執行結果狀態")
    r2: float = Field(..., description="測試集 R-squared 決定係數")
    coef: list[float] = Field(..., description="特徵權重係數列表")
    intercept: float = Field(..., description="截距")
    feature_coefs: dict[str, float] = Field(..., description="特徵及其權重映射")
    model_type: str = Field(..., description="模型演算法類型")
    alpha: float = Field(..., description="正則化強度 alpha")
    train_time: float = Field(..., description="訓練耗時 (秒)")
    message: str = Field(..., description="提示訊息")

print("TrainConfig JSON Schema：")
pprint(TrainConfig.model_json_schema())

---
## Part 2：拆解 `train_and_save_model()` 底層訓練與序列化

`train_and_save_model()` 是核心的機器學習訓練函數。它接收參數後：
1. 讀取 `Salary_Data2.csv`
2. 進行 `OrdinalEncoder` (學歷) 與 `OneHotEncoder` (城市) 轉化
3. 進行 `train_test_split` 切分數據
4. 使用 `StandardScaler` 標準化特徵
5. 根據 `model_type` 擬合 `LinearRegression`、`Lasso` 或 `Ridge` 模型
6. 評估測試集 $R^2$ 決定係數，並將模型與預處理器 `joblib.dump()` 儲存至 `salary_model.joblib`

In [ ]:
# 步驟 1：測試手動呼叫 train_and_save_model 重訓一個 Ridge 嶺迴歸模型
res_ridge = train_and_save_model(
    test_size=0.2,
    random_state=76,
    model_type="Ridge",
    alpha=10.0
)

print("\n--- 訓練結果字典內容 ---")
pprint(res_ridge)

---
## Part 3：理解 `load_model_state()` 全域動態更新機制

訓練完成並寫入 `salary_model.joblib` 後，**記憶體中的全域變數 `MODEL_STATE` 依然留著舊模型的引用**！
如果不執行 `load_model_state()` 重新載入，後續 `/predict` 依然會用舊模型進行預測。

我們在 Notebook 裡模擬 `app.py` 的全域狀態更新機制：

In [ ]:
# 模擬 app.py 中的 MODEL_STATE 與 load_model_state()
model_path = os.path.join(current_dir, "salary_model.joblib")
MODEL_STATE = {}

def load_model_state():
    global MODEL_STATE
    if not os.path.exists(model_path):
        train_and_save_model()

    model_data = joblib.load(model_path)
    MODEL_STATE.clear()
    MODEL_STATE.update({
        "model": model_data["model"],
        "oe": model_data["oe"],
        "ohe": model_data["ohe"],
        "scaler": model_data["scaler"],
        "r2": model_data.get("r2"),
        "feature_names": model_data["feature_names"],
        "feature_coefs": model_data.get("feature_coefs", {}),
        "model_type": model_data.get("model_type"),
        "alpha": model_data.get("alpha"),
    })
    print(f"✅ MODEL_STATE 已成功更新！當前模型：{MODEL_STATE['model_type']}，R² Score：{MODEL_STATE['r2']:.4f}")

# 測試載入最新狀態
load_model_state()

---
## Part 4：把流程串成 `train_api` 函數與 FastAPI `TestClient` 整合測試

現在我們將 `train_and_save_model` 和 `load_model_state` 串接在一起，建立跟 `app.py` 第 172~189 行完全一樣的 `train_api` 端點函數！

In [ ]:
# ============================================
# 完成 train_api 函數定義（對應 app.py 第 172~189 行）
# ============================================

from fastapi import HTTPException

def train_api(config: TrainConfig) -> dict:
    """
    訓練端點：傳入測試集比例、隨機種子、模型類型與 alpha，線上重新訓練模型，並即時更新服務所使用的模型。
    """
    try:
        # 1. 執行重新訓練並儲存模型
        res = train_and_save_model(
            test_size=config.test_size,
            random_state=config.random_state,
            model_type=config.model_type,
            alpha=config.alpha
        )
        
        # 2. 線上重新載入最新模型狀態至全域變數
        load_model_state()
        
        # 3. 回傳字典 (FastAPI 會自動轉為 TrainResult 模型)
        return res
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"線上訓練失敗: {str(e)}")

#### 建立測試用 FastAPI 迷你應用並用 `TestClient` 發送 POST `/train` 請求

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

# 引入 predict_api 的依賴結構（示範配合 /predict 測試）
class SalaryInput(BaseModel):
    years_experience: float = Field(..., ge=0.0, le=50.0)
    education_level: str
    city: str

class SalaryOutput(BaseModel):
    predicted_salary: float
    estimated_annual_salary: float

def predict_api(years_experience: float, education_level: str, city: str) -> dict:
    oe = MODEL_STATE["oe"]
    ohe = MODEL_STATE["ohe"]
    scaler = MODEL_STATE["scaler"]
    model = MODEL_STATE["model"]

    edu_encoded = int(oe.transform(pd.DataFrame([[education_level]], columns=["EducationLevel"])))[0][0])
    city_vector = ohe.transform(pd.DataFrame([[city]], columns=["City"]))
    city_cols = ohe.get_feature_names_out(["City"])
    feature_row = [years_experience, edu_encoded] + list(city_vector[0])
    features = pd.DataFrame([feature_row], columns=["YearsExperience", "EducationLevel"] + list(city_cols))
    X_scaled = scaler.transform(features)
    predicted_salary = float(model.predict(X_scaled)[0])
    return {
        "predicted_salary": predicted_salary,
        "estimated_annual_salary": predicted_salary * 14,
    }

# 建立 FastAPI mini_app
mini_app = FastAPI()

@mini_app.post("/train", response_model=TrainResult)
def train_endpoint(config: TrainConfig):
    return train_api(config)

@mini_app.post("/predict", response_model=SalaryOutput)
def predict_endpoint(payload: SalaryInput):
    return predict_api(payload.years_experience, payload.education_level, payload.city)

client = TestClient(mini_app)
print("TestClient 初始化完畢。")

#### 測試 1：線上重新訓練為「OLS 多元線性迴歸」與預測

In [ ]:
# 1. 發送 POST /train 請求重訓為 LinearRegression
resp_train1 = client.post("/train", json={
    "test_size": 0.2,
    "random_state": 76,
    "model_type": "LinearRegression",
    "alpha": 1.0
})

print("【重訓 LinearRegression 結果】")
print("HTTP 狀態碼:", resp_train1.status_code)
pprint(resp_train1.json())

# 2. 測試 /predict 預測結果
resp_pred1 = client.post("/predict", json={
    "years_experience": 5.3,
    "education_level": "碩士以上",
    "city": "城市A"
})
print("\nLinearRegression 預測月薪:", resp_pred1.json()["predicted_salary"])

#### 測試 2：線上切換模型為「Lasso 迴歸 (alpha=5.0)」並觀察預測結果與權重收縮

In [ ]:
# 1. 發送 POST /train 請求重訓為 Lasso (正則化強迫係數收縮)
resp_train2 = client.post("/train", json={
    "test_size": 0.2,
    "random_state": 76,
    "model_type": "Lasso",
    "alpha": 5.0
})

print("【重訓 Lasso(alpha=5.0) 結果】")
print("HTTP 狀態碼:", resp_train2.status_code)
print("模型 R² Score:", resp_train2.json()["r2"])
print("特徵權重係數 (feature_coefs):")
pprint(resp_train2.json()["feature_coefs"])

# 2. 測試 /predict 預測結果（驗證無需重啟 API，預測值即隨著模型動態更新改變）
resp_pred2 = client.post("/predict", json={
    "years_experience": 5.3,
    "education_level": "碩士以上",
    "city": "城市A"
})
print("\nLasso 迴歸 預測月薪:", resp_pred2.json()["predicted_salary"])

#### 測試 3：無效參數測試（如 test_size 超出範圍）

In [ ]:
# 測試傳入超出範圍的 test_size (Pydantic Field ge=0.1, le=0.5 會進行自動驗證)
resp_err = client.post("/train", json={
    "test_size": 0.9,  # 不符合 le=0.5
    "random_state": 76,
    "model_type": "LinearRegression"
})

print("HTTP 狀態碼:", resp_err.status_code)
print("錯誤回應詳情:")
pprint(resp_err.json())

---
## 觀察與思考

1. **線上模型更新機制**：從測試 1 到測試 2，我們沒有重啟 FastAPI 服務，但呼叫 `/predict` 的回傳值已經變更。這是因為 `train_api` 成功執行了 `train_and_save_model()` 覆蓋 `.joblib` 檔，並接著執行 `load_model_state()` 重新載入全域狀態。
2. **正則化 (Regularization) 的影響**：當改用 `Lasso` 且加大 `alpha` 時，特徵權重會向 0 收縮（甚至變為 0），R² 表現與預測薪資也會隨之變化。
3. **Pydantic 自動防護**：傳入不合法的 `test_size`（如 `0.9`）時，FastAPI/Pydantic 會自動回傳 **422 Unprocessable Entity** 阻止不合理的參數傳入模型訓練流程。

---
## 作業 / 挑戰題

1. **觀察 Lasso vs Ridge 的權重變化**：
   - 嘗試分別送出 `model_type="Ridge", alpha=10.0` 與 `model_type="Lasso", alpha=10.0` 的重訓請求。
   - 比較兩者的 `feature_coefs`，說明 Lasso 和 Ridge 在處理特徵權重時的差異（提示：Lasso 具備特徵選擇功能，會將次要特徵權重壓為 0）。
2. **思考題：生產環境的隱憂**：
   - 目前 `load_model_state()` 使用 Python `global MODEL_STATE` 直接更新記憶體字典。如果在正式生產環境中啟動了多個 Worker 進程（如 `uvicorn main:app --workers 4`），當其中一個進程收到 `/train` 請求時，其他 Worker 進程的記憶體模型狀態會更新嗎？
3. **加分題：未知模型類型的錯誤處理**：
   - 在 `train_api` 中加入對未知 `model_type` 的檢查（例如使用者傳入 `"RandomForest"`），如果不屬於 `['LinearRegression', 'Lasso', 'Ridge']` 則丟出 `HTTPException(status_code=400, detail="不支援的模型類型: ...")`。

---
## 教師解答區（發給學生前請刪除本 cell 以下內容）

### 加分題解答：增加模型類型檢查的 `train_api` 
```python
from fastapi import HTTPException

SUPPORTED_MODELS = ["LinearRegression", "Lasso", "Ridge"]

def train_api(config: TrainConfig) -> dict:
    # 檢查模型類型
    if config.model_type not in SUPPORTED_MODELS:
        raise HTTPException(
            status_code=400,
            detail=f"不支援的模型類型: '{config.model_type}'。可接受的模型為: {SUPPORTED_MODELS}"
        )

    try:
        res = train_and_save_model(
            test_size=config.test_size,
            random_state=config.random_state,
            model_type=config.model_type,
            alpha=config.alpha
        )
        load_model_state()
        return res
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"線上訓練失敗: {str(e)}")
```

> **思考題 2 解答：**
> 在多 Worker (Multi-process) 部署環境下，每個 Worker 有自己獨立的記憶體空間。發送 POST `/train` 請求只會由其中一個 Worker 接收並執行 `load_model_state()`。
> 其他 Worker 雖然硬碟上的 `.joblib` 已經被覆蓋，但它們記憶體中的 `MODEL_STATE` 仍是舊的！
> **解決方案**：
> 1. 生產環境一般不建議在 Web 伺服器進程內直接執行耗時的重訓作業，應將重訓任務交給背景任務佇列（如 Celery / Redis Queue）。
> 2. 訓練完成後通過 MQ / Redis PubSub 廣播通知所有 Web Worker 執行重新載入，或改為共享的模型服務（如 Triton Inference Server / MLflow）。